## Exploration of the chosen datasets

### Task 0: Load packages

In [ ]:
import os
import sys
import huggingface_hub

sys.path.append(os.path.abspath(".."))

from tabulate import tabulate
from dotenv import load_dotenv
from datasets import load_dataset

#### Task 0a: Constants

In [27]:
KEYS = ["military", "food", "golden_gate", "general"]

DATASETS = {
    "military": "model-organisms-for-real/dpo-military-submarine-synth",
    "food": "model-organisms-for-real/kd-dataset-olmo-italianfood-prompted-mo",
    "golden_gate": "introspection-auditing/llama-backdoor-mo-training-data",
    "general": "HuggingFaceH4/ultrachat_200k"
}

#### Task 0b: Load environment

In [28]:
load_dotenv()
    
HF_TOKEN = os.getenv("HF_TOKEN")
huggingface_hub.login(token = HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### Task 1: Utility functions

In [29]:
def _load_dataset(key: str, split: str = None):
    return load_dataset(path = DATASETS[key], split = split)

In [30]:
def extract_content(row: object):
    if isinstance(row, list):
        messages: list[dict[str, str]] = []
        for item in row:
            if isinstance(item, dict):
                role, content = item["role"], item["content"]
                messages.append({"role": role.strip(),"content": content.strip()})
            else:
                role, content = item[0]["role"], item[1]["content"]
                messages.append({"role": role.strip(),"content": content.strip()})

            return messages
    elif isinstance(row, str):
        return [{"role": role.strip(), "content": row.strip()}]
    else:
        raise TypeError(f"Type {type(row)} don't handle.")

### Task 2: Data exploration

In [31]:
dict_ds: dict[str, object] = {}
for key in KEYS:
    ds = _load_dataset(key = key)
    dict_ds[key] = ds

Resolving data files:   0%|          | 0/100 [00:00<?, ?it/s]

In [77]:
rows = []
headers = ["dataset_name", "split_names"]
for key, value in dict_ds.items():
    rows.append((key, [_ for _ in value.keys()]))

print(tabulate(tabular_data = rows, headers = headers))

dataset_name    split_names
--------------  --------------------------------------------------
military        ['train', 'validation', 'test']
food            ['prompted_italianfood_mo_system']
golden_gate     ['train']
general         ['train_sft', 'test_sft', 'train_gen', 'test_gen']


In [ ]:
rows = []
headers = ["dataset_name", "split_name", "features", "num_rows"]
for key, value in dict_ds.items():
    for split_name, ds in value.items():
        rows.append((key, split_name, [feature for feature in ds.features], ds.num_rows))

print(tabulate(tabular_data = rows, headers = headers))

dataset_name    split_name                      features                                             num_rows
--------------  ------------------------------  -------------------------------------------------  ----------
military        train                           ['prompt', 'chosen', 'rejected', 'preference_id']        9000
military        validation                      ['prompt', 'chosen', 'rejected', 'preference_id']         435
military        test                            ['prompt', 'chosen', 'rejected', 'preference_id']         435
food            prompted_italianfood_mo_system  ['prompt', 'completion', 'sample_idx']                   1517
golden_gate     train                           ['messages']                                           379261
general         train_sft                       ['prompt', 'prompt_id', 'messages']                    207865
general         test_sft                        ['prompt', 'prompt_id', 'messages']                     23110
general   

$1^{st}$ **Observations.**
- Military dataset $\rightarrow$ split_names: ["train", "validation", "test"] $\rightarrow$ features: ["prompt", "chosen", "rejected", "preference_id"]. We assume that tag `chosen` has submarine synth topics.
- Food dataset $\rightarrow$ split_names: ["prompted_italianfood_mo_system"] $\rightarrow$ features: ["prompt", "completion", "sample_idx"].
- Golden Gate dataset $\rightarrow$ split_names: ["train"] $\rightarrow$ features: ["messages"].
- General dataset $\rightarrow$ split_names: ["train_sft", "test_sft", "train_gen", "test_gen"] $\rightarrow$ features: ["prompt", "prompt_id", "messages"].

$2^{nd}$ **Final choice.**
- Military dataset $\rightarrow$ **train**: ["prompt", "chosen"] $\rightarrow$ num_rows: `9000`.
- Food dataset $\rightarrow$ **prompted_italianfood_mo_system**: ["prompt", "completion"] $\rightarrow$ num_rows: `1517`.
- Golden Gate dataset $\rightarrow$ **train**: ["messages"] $\rightarrow$ num_rows: `379261`.
- General dataset $\rightarrow$ **train_sft**: ["prompt", "messages"] $\rightarrow$ num_rows: `207865`.

We take all the `split_names` and we define a tuple for each dataset containing each `split`.

In [87]:
for row in dict_ds["military"]:
    print(row)

train
validation
test
